# Neural Networks Practice Lab
## One Neuron → Forward Pass → Backpropagation → MLP → Keras → Independent Task

This lab follows the same learning path you already saw in class. Work through the
theory questions first, then complete the coding tasks step by step.

**Learning path**
1. Answer the theory (multiple choice) questions.
2. Program the forward pass of one neuron from scratch.
3. Program the backward pass and update the parameters from scratch.
4. Extend the same logic to a small Multi-Layer Perceptron (MLP).
5. See how TensorFlow/Keras automates the same process.
6. Build and train an MLP independently.

> For the manual section we use **Sigmoid + MSE** because the derivatives are easy to follow. For practical binary classification with Keras, **Binary Cross-Entropy** is the usual loss.


# Part 0 — Theory Questions (Multiple Choice)

Choose the correct option for each question and write the answer in the space provided.

---

**Q1. What does `z = w·x + b` represent in a single neuron?**

- A) The final prediction of the neuron
- B) The raw weighted sum before activation
- C) The loss of the neuron
- D) The gradient of the weights

**Answer:** 'B'

---

**Q2. Why is the Sigmoid function applied to `z`?**

- A) To make the computation faster
- B) To convert the raw score into a value between 0 and 1
- C) To calculate the loss directly
- D) To remove the bias term

**Answer:** 'B'

---

**Q3. In gradient descent, what does the learning rate control?**

- A) The number of layers in the network
- B) The size of the step taken when updating a parameter
- C) The number of training samples used
- D) The activation function used

**Answer:** 'B'

---

**Q4. Why does backpropagation compute gradients starting from the output layer and move backward?**

- A) Because the input layer has no parameters to update
- B) Because the loss is defined at the output, and the chain rule propagates that error backward through the layers
- C) Because it is faster to compute gradients in reverse order regardless of where the loss is
- D) Because hidden layers do not affect the final prediction

**Answer:** 'B'

---

**Q5. What is the main advantage of using Keras/TensorFlow instead of writing forward and backward passes manually?**

- A) It removes the need for a loss function
- B) It automatically computes gradients and updates parameters, while still following the same math
- C) It changes the underlying mathematics of neural networks
- D) It only works for classification problems

**Answer:** 'B'

---


# Part 1 — Core Equations (Reference)

### Forward pass of one neuron
- Weighted sum: `z = w·x + b`
- Activation: `a = sigmoid(z)`
- Sigmoid: `sigmoid(z) = 1 / (1 + exp(-z))`

### Loss used in the manual section
- `L = (a - y)^2`

### Derivatives used in backpropagation
- `dL/da = 2(a-y)`
- `da/dz = a(1-a)`
- `dL/dz = (dL/da)(da/dz)`
- `dL/dw = (dL/dz)x`
- `dL/db = dL/dz`

### Gradient-descent update
- `parameter_new = parameter_old - learning_rate × gradient`

**Big picture:** Forward Pass → Loss → Backward Pass → Update


In [1]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

# Part 2 — One Neuron: Forward Pass From Scratch

Use one neuron with two inputs. The goal is to understand exactly what happens before moving to multiple neurons.


In [2]:
x = np.array([0.3, 0.9])
w = np.array([0.5, -0.4])
b = 0.2
y = 0.0

## Task 2.1 — Implement Sigmoid
Complete the function below.

In [3]:
def sigmoid(z):
    # TODO: implement the sigmoid function
    return 1 / (1 + np.exp(-z))

## Task 2.2 — Implement the Forward Pass
Compute `z`, then apply Sigmoid to obtain `a`. Return both values.


In [4]:
def forward_single(x, w, b):
    # TODO: compute z (weighted sum) and a (activation), then return them
    z = x * w + b
    a = sigmoid(z)
    return z, a

In [5]:
z, a = forward_single(x, w, b)
print("z =", z)
print("prediction a =", a)

z = [ 0.35 -0.16]
prediction a = [0.5866 0.4601]


### Checkpoint
1. What is the difference between `z` and `a`?
2. Which values are learnable parameters?

*(Write your answers here.)*


# Part 3 — One Neuron: Backward Pass From Scratch

Now calculate how the loss changes with respect to the weights and bias.


## Task 3.1 — Implement the MSE Loss

In [6]:
def mse_loss(a, y):
    # TODO: implement the mean squared error for a single example
    L = (a - y) ** 2
    return L

In [7]:
loss = mse_loss(a, y)
print("Loss =", loss)

Loss = [0.3441 0.2117]


## Task 3.2 — Compute Gradients
Use the chain rule:
`dL/da → da/dz → dL/dz → dW and db`


In [8]:
def backward_single(x, y, a):
    # TODO: compute dL_da, da_dz, dL_dz, then dW and db, and return them
    dL_da = 2 * (a - y)

    da_dz = a * (1 - a)

    dL_dz = (dL_da) * (da_dz)
    dL_dw = (dL_dz) * x
    dL_db = dL_dz

    return dL_dw, dL_db

In [9]:
dW, db = backward_single(x, y, a)
print("dW =", dW)
print("db =", db)

dW = [0.0854 0.2057]
db = [0.2845 0.2286]


## Task 3.3 — Update the Parameters

In [10]:
learning_rate = 0.1
# TODO: compute w_new and b_new using the gradient descent update rule

w_new = w - learning_rate * dW
b_new = b - learning_rate * db

print("Old w:", w, " New w:", w_new)
print("Old b:", b, " New b:", b_new)

Old w: [ 0.5 -0.4]  New w: [ 0.4915 -0.4206]
Old b: 0.2  New b: [0.1715 0.1771]


## Task 3.4 — Did the Update Reduce the Loss?

In [11]:
# TODO: run the forward pass again with w_new, b_new and compare the new loss to the old one
z_new, a_new = forward_single(x, w_new, b_new)
new_loss = mse_loss(a_new, y)
print("Old loss:", loss)
print("New loss:", new_loss)

Old loss: [0.3441 0.2117]
New loss: [0.3353 0.2023]


# Part 4 — From One Neuron to a Multi-Layer Perceptron

Architecture: **2 inputs → 2 hidden neurons → 1 output neuron**

The logic is exactly the same; we now repeat it across layers.


In [12]:
x = np.array([0.0, 1.0])
y = 0.0
W1 = np.array([[0.2, -0.1],[0.3, 0.4]])
b1 = np.array([0.1, -0.2])
W2 = np.array([[0.4],[-0.3]])
b2 = np.array([0.05])

## Task 4.1 — MLP Forward Pass From Scratch
Compute hidden `z1`, hidden activation `a1`, output `z2`, then `y_hat`.


In [ ]:
def forward_mlp(x, W1, b1, W2, b2):
    # TODO: compute z1, a1, z2, y_hat, then build cache and return y_hat, cache
    z1 = x @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    y_hat = sigmoid(z2)

    cache = {"x": x, "z1": z1, "a1": a1, "z2": z2, "y_hat": y_hat}

    return y_hat, cache


In [ ]:
y_hat, cache = forward_mlp(x, W1, b1, W2, b2)
print("Hidden activation:", cache["a1"])
print("Prediction:", y_hat)

## Task 4.2 — MLP Backward Pass From Scratch

**Output layer**
- `delta2 = 2(y_hat-y) × y_hat(1-y_hat)`
- `dW2 = a1 × delta2`
- `db2 = delta2`

**Hidden layer**
- Send the output error backward through `W2`
- Multiply by the hidden sigmoid derivative
- Compute `dW1` and `db1`


In [ ]:
def backward_mlp(y, cache, W2):
    x = cache["x"]
    a1 = cache["a1"]
    y_hat = cache["y_hat"]
    # TODO: compute delta2, dW2, db2, da1, delta1, dW1, db1 and return dW1, db1, dW2, db2
    pass

In [ ]:
dW1, db1, dW2, db2 = backward_mlp(y, cache, W2)
print("dW1:", dW1)
print("db1:", db1)
print("dW2:", dW2)
print("db2:", db2)

### Explain before moving on
1. Why does backpropagation start from the output?
2. What does a gradient tell us?
3. Why does the hidden layer receive its error from the next layer?

*(Write your answers here.)*


# Part 5 — The Same Idea with TensorFlow / Keras

Keras automates the matrix operations, forward propagation, loss calculation, automatic differentiation, backpropagation, and parameter updates.


In [ ]:
import tensorflow as tf
tf.random.set_seed(7)
print(tf.__version__)

In [ ]:
X_train = np.array([[0.05,0.10],[0.15,0.22],[0.28,0.18],[0.35,0.30],
                     [0.60,0.55],[0.68,0.72],[0.80,0.60],[0.90,0.85]], dtype=np.float32)
y_train = np.array([0,0,0,0,1,1,1,1], dtype=np.float32)

## Guided Keras Example
Architecture: **2 inputs → 3 hidden neurons → 1 output neuron**

This part is provided as an example — read it and run it to see how the pieces connect.


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(3, activation="sigmoid"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.1),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(X_train, y_train, epochs=300, verbose=0)
loss_value, accuracy = model.evaluate(X_train, y_train, verbose=0)
print("Loss:", round(loss_value,4))
print("Accuracy:", round(accuracy,4))

### Connect the code to the theory
- `Dense(...)` performs the weighted sums and activations.
- `loss=...` defines how error is measured.
- TensorFlow computes gradients automatically.
- The optimizer updates the parameters.
- `model.fit()` repeats the training loop across epochs.


# Part 6 — Independent Student Task

Build your own MLP for a non-linear binary classification dataset.


In [ ]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_task, y_task = make_circles(n_samples=500, noise=0.12, factor=0.5, random_state=7)
X_train_task, X_test_task, y_train_task, y_test_task = train_test_split(
    X_task, y_task, test_size=0.2, random_state=7, stratify=y_task
)
scaler = StandardScaler()
X_train_task = scaler.fit_transform(X_train_task)
X_test_task = scaler.transform(X_test_task)

plt.figure(figsize=(6,5))
plt.scatter(X_train_task[:,0], X_train_task[:,1], c=y_train_task, s=25)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Independent Task Dataset")
plt.show()

## Task 6.1 — Build the Model
Build a Keras `Sequential` model suitable for this dataset (choose the number of hidden
layers/neurons yourself). Compile it with an appropriate optimizer, loss, and metric.


In [ ]:
# TODO: build your Sequential model here


## Task 6.2 — Train the Model

In [ ]:
# TODO: train model_task on X_train_task, y_train_task


## Task 6.3 — Evaluate the Model

In [ ]:
# TODO: evaluate model_task on X_test_task, y_test_task and print loss/accuracy


### Reflection
1. How did you choose the number of hidden neurons/layers?
2. How does the test accuracy compare to the training accuracy? What does that tell you?

*(Write your answers here.)*
